In [22]:
# Sezione: import delle librerie necessarie
#
# Si usano:
# - networkx per la rappresentazione del grafo
# - numpy per i vettori di attivazione
# - deque per le visite in ampiezza
# - matplotlib per eventuali visualizzazioni

import networkx as nx
import numpy as np
from collections import deque
import matplotlib.pyplot as plt
import time

# Helper

In [2]:
# Sezione: funzioni ausiliarie
#
# - conversione di un seed set in vettore binario di attivazione
# - calcolo della cardinalità finale e della traiettoria della diffusione

def make_vector(seed_set, n):
    x = np.zeros(n)
    x[list(seed_set)] = 1
    return x

# Diffusion Model

In [3]:
# Sezione: dinamica di diffusione

def connected_component_update(g, x, thetas):
    new_x = x.copy()
    n = len(x)

    active = set(np.where(x > 0)[0])

    for v in range(n):
        if x[v] == 1:
            continue

        reachable = 0
        visited = set()

        for u in g.neighbors(v):
            if u in active and u not in visited:
                comp = nx.node_connected_component(g.subgraph(active), u)
                reachable += len(comp)
                visited |= comp

        if reachable >= thetas[v] - 1:
            new_x[v] = 1

    return new_x


def connected_component_spread(g, x0, thetas, max_t=1000):
    x = x0.copy()
    spread_hist = [int(np.sum(x))]

    for _ in range(max_t):
        new_x = connected_component_update(g, x, thetas)

        if np.array_equal(new_x, x):
            break

        x = new_x
        spread_hist.append(int(np.sum(x)))

    return int(np.sum(x)), spread_hist, x



# Gamma

In [4]:
# Sezione: costruzione delle strutture per Gamma

# Sezione: Gamma(theta_i, S)
#
# Per ogni livello di soglia distinto theta_i, si considera il sottografo
# indotto dai vertici con soglia al più theta_i.
#
# La quantità Gamma(theta_i, S) è l'unione delle componenti connesse,
# in tale sottografo, che contengono almeno un vertice di S.

def build_gamma_data(g, thetas):
    n = g.number_of_nodes()

    levels = sorted(set(thetas))
    comp_id_by_level = []
    comp_size_by_level = []

    for level in levels:
        allowed = [v for v in g.nodes() if thetas[v] <= level]
        H = g.subgraph(allowed)

        comp_id = [-1] * n
        comp_size = {}

        cid = 0
        for comp in nx.connected_components(H):
            comp = list(comp)
            for v in comp:
                comp_id[v] = cid
            comp_size[cid] = len(comp)
            cid += 1

        comp_id_by_level.append(comp_id)
        comp_size_by_level.append(comp_size)

    return levels, comp_id_by_level, comp_size_by_level


    # : calcolo di |Gamma(theta_i, S)|

def gamma_size(g, thetas, S, level_idx, levels, comp_id_by_level, comp_size_by_level):
    level = levels[level_idx]

    touched_components = set()
    high_nodes = set()

    for v in S:
        if thetas[v] <= level:
            cid = comp_id_by_level[level_idx][v]
            if cid != -1:
                touched_components.add(cid)
        else:
            high_nodes.add(v)
            for u in g.neighbors(v):
                cid = comp_id_by_level[level_idx][u]
                if cid != -1:
                    touched_components.add(cid)

    total = len(high_nodes)
    total += sum(comp_size_by_level[level_idx][cid] for cid in touched_components)

    return total

# Funzione submodulare F

In [5]:
# Sezione: implementazione della funzione submodulare f

def f_value(g, thetas, S, levels, comp_id_by_level, comp_size_by_level):
    l = len(levels)

    if l == 1:
        return max(0, min(len(S), levels[0] - 1))

    total = 0
    for i in range(l - 1):
        target = levels[i + 1] - 1
        total += min(
            gamma_size(g, thetas, S, i, levels, comp_id_by_level, comp_size_by_level),
            target
        )

    return total


def f_target(levels):
    l = len(levels)

    if l == 1:
        return max(0, levels[0] - 1)

    return sum(levels[i + 1] - 1 for i in range(l - 1))

# Greedy Algorithm

In [23]:
def greedy_seed_set(g, thetas, levels, comp_id_by_level, comp_size_by_level):
    start = time.perf_counter()
    S = set()
    target = f_target(levels)
    remaining = set(g.nodes())

    while f_value(g, thetas, S, levels, comp_id_by_level, comp_size_by_level) < target:
        best_v = None
        best_gain = -1

        current_value = f_value(g, thetas, S, levels, comp_id_by_level, comp_size_by_level)

        for v in remaining:
            new_value = f_value(g, thetas, S | {v}, levels, comp_id_by_level, comp_size_by_level)
            gain = new_value - current_value

            if gain > best_gain:
                best_gain = gain
                best_v = v

        if best_v is None or best_gain <= 0:
            break

        S.add(best_v)
        print(f"Added {best_v}, gain: {best_gain}, f(S): {f_value(g, thetas, S, levels, comp_id_by_level, comp_size_by_level)}, time elapsed: {time.perf_counter() - start:.2f}s")
        remaining.remove(best_v)

    return S

# Connessione del Seedset

In [7]:
# Sezione: shortest path e connessione del seed set

def shortest_path_to_set(g, source, targets):
    if source in targets:
        return [source]

    visited = {source}
    parent = {source: None}
    queue = deque([source])

    while queue:
        u = queue.popleft()

        if u in targets:
            path = []
            while u is not None:
                path.append(u)
                u = parent[u]
            return path[::-1]

        for w in g.neighbors(u):
            if w not in visited:
                visited.add(w)
                parent[w] = u
                queue.append(w)

    return []


def connect_seed_set(g, D):
    D = list(D)

    if not D:
        return set()

    A = {D[0]}

    for v in D[1:]:
        path = shortest_path_to_set(g, v, A)
        A.update(path)

    return A

# Wrapper

In [8]:
def TD_orlogn(g, thetas, verbose=True):
    levels, comp_id_by_level, comp_size_by_level = build_gamma_data(g, thetas)

    D = greedy_seed_set(g, thetas, levels, comp_id_by_level, comp_size_by_level)
    A = connect_seed_set(g, D)

    spread, history, x = connected_component_spread(
        g,
        make_vector(A, g.number_of_nodes()),
        thetas
    )

    success = (spread == g.number_of_nodes())

    if verbose:
        print("Nodes:", g.number_of_nodes())
        print("Thresholds:", sorted(set(thetas)))
        print("D:", sorted(D))
        print("D size:", len(D))
        print("Seed set:", sorted(A))
        print("Seed set size:", len(A))
        print("Spread:", spread)
        print("Success:", success)

    return {
        "D": D,
        "A": A,
        "spread": spread,
        "history": history,
        "x": x,
        "success": success,
        "levels": levels
    }

# G&L

In [12]:
from typing import Iterable, Mapping, Optional 
import numpy as np
import networkx as nx

def create_pa_graph(
    n_nodes: int,
    c: int,
    seed: Optional[int] = None,
    init_nodes: Optional[int] = None,
    init_mode: str = "complete",
) -> tuple[nx.Graph, dict[int, int]]:
    if c <= 0:
        raise ValueError("c must be a positive integer.")
    if init_nodes is None:
        init_nodes = max(2, c)
    if not (2 <= init_nodes <= n_nodes):
        raise ValueError("init_nodes must satisfy 2 <= init_nodes <= n_nodes")

    rng_graph = np.random.default_rng(seed)
    rng_theta = np.random.default_rng(seed)

    if init_mode == "complete":
        g = nx.complete_graph(init_nodes)
    elif init_mode == "tree":
        g = nx.random_tree(init_nodes, seed=seed)
    else:
        raise ValueError("init_mode must be 'complete' or 'tree'")

    m_choices = (1, 2, 3, 4)

    for u in range(init_nodes, n_nodes):
        g.add_node(u)
        m = int(rng_graph.choice(m_choices))
        existing = np.array([v for v in g.nodes if v != u], dtype=int)
        m = min(m, len(existing))
        degrees = np.array([g.degree(v) for v in existing], dtype=float)
        probs = degrees / degrees.sum()
        targets = rng_graph.choice(existing, size=m, replace=False, p=probs)
        for v in targets:
            g.add_edge(u, int(v))

    k_max = int(np.ceil(n_nodes / c))
    values = list(range(max(2, c), k_max * c + 1, c))
    theta = {v: int(rng_theta.choice(values)) for v in g.nodes()}
    nx.set_node_attributes(g, theta, name="theta")
    return g, theta

In [14]:
thetas200

{0: 19,
 1: 156,
 2: 132,
 3: 89,
 4: 88,
 5: 172,
 6: 19,
 7: 140,
 8: 42,
 9: 20,
 10: 106,
 11: 196,
 12: 148,
 13: 153,
 14: 144,
 15: 158,
 16: 104,
 17: 27,
 18: 169,
 19: 91,
 20: 101,
 21: 75,
 22: 38,
 23: 186,
 24: 157,
 25: 130,
 26: 82,
 27: 165,
 28: 110,
 29: 90,
 30: 91,
 31: 47,
 32: 20,
 33: 112,
 34: 178,
 35: 14,
 36: 172,
 37: 166,
 38: 57,
 39: 127,
 40: 34,
 41: 152,
 42: 141,
 43: 72,
 44: 15,
 45: 195,
 46: 90,
 47: 179,
 48: 136,
 49: 156,
 50: 153,
 51: 40,
 52: 74,
 53: 94,
 54: 101,
 55: 10,
 56: 110,
 57: 32,
 58: 149,
 59: 137,
 60: 185,
 61: 150,
 62: 74,
 63: 194,
 64: 83,
 65: 66,
 66: 182,
 67: 75,
 68: 17,
 69: 95,
 70: 160,
 71: 39,
 72: 94,
 73: 27,
 74: 138,
 75: 96,
 76: 67,
 77: 47,
 78: 114,
 79: 135,
 80: 189,
 81: 88,
 82: 33,
 83: 167,
 84: 127,
 85: 141,
 86: 21,
 87: 64,
 88: 154,
 89: 167,
 90: 88,
 91: 162,
 92: 169,
 93: 79,
 94: 180,
 95: 59,
 96: 49,
 97: 137,
 98: 128,
 99: 29,
 100: 167,
 101: 41,
 102: 162,
 103: 3,
 104: 160,
 105:

In [35]:
N = 2000
GRAPH_SEED = 42
C = 20

g200, theta_dict_200 = create_pa_graph(
    n_nodes=N,
    c=C,
    seed=GRAPH_SEED,
    init_nodes=5,
    init_mode="complete",
)

thetas200 = np.array([theta_dict_200[i] for i in range(N)], dtype=int)

print("Distinct thresholds in instance:", sorted(set(thetas200)))
print("Min theta:", thetas200.min())
print("Max theta:", thetas200.max())

Distinct thresholds in instance: [np.int64(20), np.int64(40), np.int64(60), np.int64(80), np.int64(100), np.int64(120), np.int64(140), np.int64(160), np.int64(180), np.int64(200), np.int64(220), np.int64(240), np.int64(260), np.int64(280), np.int64(300), np.int64(320), np.int64(340), np.int64(360), np.int64(380), np.int64(400), np.int64(420), np.int64(440), np.int64(460), np.int64(480), np.int64(500), np.int64(520), np.int64(540), np.int64(560), np.int64(580), np.int64(600), np.int64(620), np.int64(640), np.int64(660), np.int64(680), np.int64(700), np.int64(720), np.int64(740), np.int64(760), np.int64(780), np.int64(800), np.int64(820), np.int64(840), np.int64(860), np.int64(880), np.int64(900), np.int64(920), np.int64(940), np.int64(960), np.int64(980), np.int64(1000), np.int64(1020), np.int64(1040), np.int64(1060), np.int64(1080), np.int64(1100), np.int64(1120), np.int64(1140), np.int64(1160), np.int64(1180), np.int64(1200), np.int64(1220), np.int64(1240), np.int64(1260), np.int64(12

In [36]:
res200 = TD_orlogn(g200, thetas200, verbose=True)

Added 4, gain: 83059, f(S): 83059, time elapsed: 0.10s
Added 1, gain: 857, f(S): 83916, time elapsed: 0.53s
Added 2, gain: 794, f(S): 84710, time elapsed: 1.51s
Added 7, gain: 670, f(S): 85380, time elapsed: 2.99s
Added 347, gain: 517, f(S): 85897, time elapsed: 4.81s
Added 10, gain: 436, f(S): 86333, time elapsed: 6.84s
Added 323, gain: 421, f(S): 86754, time elapsed: 9.08s
Added 27, gain: 413, f(S): 87167, time elapsed: 11.46s
Added 324, gain: 362, f(S): 87529, time elapsed: 14.09s
Added 136, gain: 349, f(S): 87878, time elapsed: 16.77s
Added 100, gain: 348, f(S): 88226, time elapsed: 19.64s
Added 24, gain: 346, f(S): 88572, time elapsed: 22.72s
Added 174, gain: 346, f(S): 88918, time elapsed: 25.96s
Added 91, gain: 327, f(S): 89245, time elapsed: 29.35s
Added 207, gain: 324, f(S): 89569, time elapsed: 32.99s
Added 564, gain: 315, f(S): 89884, time elapsed: 36.77s
Added 318, gain: 309, f(S): 90193, time elapsed: 40.69s
Added 3, gain: 287, f(S): 90480, time elapsed: 44.73s
Added 140, 